# OMonte Carlo Exploring Start (Sutton 5.3)

In [77]:
import numpy as np

In [78]:
class GridWorldEnv:
    def __init__(self, size,target_pos):
        self.action_space = ["UP","RIGHT","DOWN","LEFT"]
        self.height = size[1]
        self.width = size[0]
        self.agent_pos = (0,0)
        self.target_pos = target_pos

    def reset(self):
        self.agent_pos = (np.random.randint(0,self.width),np.random.randint(0,self.height))
        return self.agent_pos

    def step(self,action):
        action_to_change = {"UP":(0,-1) ,"RIGHT":(1,0), "DOWN":(0,1), "LEFT":(-1,0)}
        change = action_to_change[self.action_space[action]]
        previous_pos = self.agent_pos
        possible_next_pos = (
                                self.agent_pos[0] + change[0],
                                self.agent_pos[1] + change[1]
                            ) 
        x, y = possible_next_pos

        if 0 <= x < self.width and 0 <= y < self.height:
            self.agent_pos = possible_next_pos

            if self.agent_pos == self.target_pos:
                reward = 5
                done = True
            else:
                reward = -1
                done = False

        else:
            self.agent_pos = previous_pos
            reward = -2
            done = False
        return self.agent_pos, reward, done

In [79]:
class MCAgent:
    def __init__(self, env, gamma=0.99,epsilon=0.1):
        self.env = env
        self.n_actions = len(env.action_space)
        self.state_space = [
                (x, y)
                for x in range(env.width)
                for y in range(env.height)
            ]
        self.Q = {s:np.zeros(self.n_actions) for s in self.state_space}
        self.N = {s:np.zeros(self.n_actions) for s in self.state_space}
        self.gamma = gamma
        self.epsilon = epsilon
    def choose_action(self, state):
        if (np.random.rand() < self.epsilon):
            return np.random.randint(self.n_actions)
        else:
            return np.argmax(self.Q[state])

    def generate_episode(self, max_steps=100):
        # episode history
        episode = []
        # exploring start
        state = self.env.reset()
        for i in range(max_steps):
            state = self.env.agent_pos
            action = self.choose_action(self.env.agent_pos)
            next_state, reward, done = self.env.step(action)
            episode.append((state,action, reward))
            if(done):
                break
        return episode        
            
    def update_Q_from_episode(self,episode):
        G = 0
        visited_state_actions = set()

        # Go backward through the episode
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]

            G = reward + self.gamma * G

            # First-visit Monte Carlo
            if (state, action) not in visited_state_actions:
                visited_state_actions.add((state, action))

                self.N[state][action] += 1

                alpha = 1 / self.N[state][action]

                self.Q[state][action] += alpha * (G - self.Q[state][action])

    def train(self, num_episode=10000):
        for i in range(num_episode):
            episode = self.generate_episode(100)
            self.update_Q_from_episode(episode)
        return self.Q
        

In [80]:
env = GridWorldEnv((5,5),(4,4))
agent = MCAgent(env, gamma=0.99,epsilon = 0.1)
Q = agent.train(num_episode=10000)

In [81]:
for y in range(env.height):
    row = []
    for x in range(env.width):
        state = (x, y)

        if state == env.target_pos:
            row.append("T")
        else:
            best_action = np.argmax(agent.Q[state])
            row.append(env.action_space[best_action])
    print(row)

['LEFT', 'RIGHT', 'RIGHT', 'DOWN', 'LEFT']
['DOWN', 'DOWN', 'RIGHT', 'DOWN', 'DOWN']
['DOWN', 'DOWN', 'RIGHT', 'DOWN', 'DOWN']
['DOWN', 'DOWN', 'DOWN', 'DOWN', 'DOWN']
['RIGHT', 'RIGHT', 'RIGHT', 'RIGHT', 'T']
